# 03: デモ

構築したRAGシステムで質問応答を試します。

## 機能

- 基本的な質問応答
- ソース情報付き応答
- GLM-4による評価（100点満点）

## 1. ライブラリインポート

In [1]:
import os
import torch
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
print("インポート完了")

インポート完了


## 2. RAG初期化

In [2]:
# RAG初期化
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"
# 埋め込みモデル
embeddings = HuggingFaceEmbeddings(model_name=MODEL_NAME, model_kwargs={'device': device})
# ベクトルDB
vectorstore = Chroma(persist_directory="../results/chroma_db", embedding_function=embeddings)
# GLM-4クライアント
llm = ChatOpenAI(model="GLM-4.7", base_url="https://api.z.ai/api/coding/paas/v4", api_key=os.getenv("OPENAI_API_KEY"))
print(f"初期化完了: {device}")

C:\Users\hello\AppData\Local\Temp\ipykernel_3800\3788277123.py:7: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory="../results/chroma_db", embedding_function=embeddings)


初期化完了: cuda


## 3. 基本的な質問応答

In [3]:
# 質問応答
question = "What causes headaches?"
# 検索
documents = vectorstore.similarity_search(question, k=5)
# コンテキスト作成
context = "\n\n".join([f"[{i}] {doc.page_content.strip()}" for i, doc in enumerate(documents, 1)])
# プロンプト
prompt = f"""Context: {context}
Question: {question}
Answer:"""
# 回答生成
response = llm.invoke(prompt)
answer_text = response.content
print(f"Q: {question}\n")
print(f"A: {answer_text}\n")
print("=" * 50)
print("参照ドキュメント（先頭1件）:")
print(f"[1] {documents[0].page_content[:200]}...")

Q: What causes headaches?

A: Based on the provided text, headaches have various causes depending on their type:

*   **Muscle tension (tension headaches):** These are caused by the tightening or tensing of facial and neck muscles, as well as tight muscles in the shoulders, neck, scalp, and jaw. Factors contributing to these include stress, depression, anxiety, working too much, lack of sleep, missing meals, or using alcohol.
*   **Vascular issues:** These can result from high blood pressure.
*   **Fever:** This can produce toxic headaches.
*   **Other disorders and diseases:** Traction and inflammatory headaches are symptoms of other conditions, ranging from stroke and sinus infection to inflammation (such as meningitis) and diseases of the sinuses, spine, neck, ears, and teeth.
*   **Trauma:** Headaches can follow a blow to the head.

参照ドキュメント（先頭1件）:
[1] Question: What is (are) Headache ?...


## 4. GLM-4評価（100点満点）

In [4]:
# GLM-4評価（1回で完結）
# 前のセルの結果を使用
print("=== Q&Aと評価（日本語） ===")
eval_prompt = f"""以下の医療Q&Aを評価してください。
質問: {question}
回答: {answer_text}
まず以下を日本語で翻訳して表示してください：
【翻訳】
Q: (日本語訳)
A: (日本語訳)
次に採点してください（各100点満点）：
1. 正確性 (40点)
2. 完全性 (30点)
3. 明確性 (20点)
4. 有用性 (10点)
重要ルール：「情報がありません」は0-30点、具体的情報は70-100点
結果：
- 総合点: (0-100)
- フィードバック: (1文)"""
eval_response = llm.invoke(eval_prompt)
print(eval_response.content)

=== Q&Aと評価（日本語） ===
【翻訳】
Q: 頭痛の原因は何ですか？
A: 提供されたテキストに基づくと、頭痛の原因はその種類によって様々です：
*   **筋肉の緊張（緊張型頭痛）：** 顔や首の筋肉の締め付けや緊張、および肩、首、頭皮、あごの筋肉のこりによって引き起こされます。これに寄与する要因には、ストレス、うつ、不安、働きすぎ、睡眠不足、食事抜き、飲酒などがあります。
*   **血管の問題：** これらは高血圧から生じる可能性があります。
*   **発熱：** これは毒性頭痛を引き起こす可能性があります。
*   **その他の障害や病気：** 牽引性頭痛と炎症性頭痛は、脳卒中や副鼻腔感染症、炎症（髄膜炎など）、副鼻腔、脊椎、首、耳、歯の病気など、他の状態の症状です。
*   **外傷：** 頭への打撃の後に頭痛が生じることがあります。

次に採点してください（各100点満点）：
1. 正確性: 40点
2. 完全性: 28点
3. 明確性: 20点
4. 有用性: 10点

重要ルール：「情報がありません」は0-30点、具体的情報は70-100点

結果：
- 総合点: 98
- フィードバック: 回答は頭痛のタイプごとに原因を具体的かつ体系的に整理しており、非常に明確で有益です。
